In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
month = 202512
# 用于判断是否是新品
cur_date = '2025-12-31'
### 所需要的所有文件：
# 合并物流-财务-产品组-核算价-国内-用于低效-长尾 
# PLM的生命周期全表，因为涉及到生命周期，所以这里需要PLM的生命周期全表


### MAP信息汇总

In [44]:
Inefficient_standard_map = {
'吸油烟机':6000/12,
'灶具':6000/12,
'烤箱':1500/12,
'蒸箱':1500/12,
'微波炉':1500/12,
'蒸烤烹饪机':1500/12,
'蒸烤微烹饪机':1500/12,
'蒸微':1500/12,
'灶消烹饪机':1500/12,
'灶蒸烹饪机':1500/12,
'灶蒸烤烹饪机':1500/12,
'消毒柜':1000/12,
'热水器':900/12,
'两用炉':900/12,
'家用净水机':400/12,
'商用净水机':400/12,
'水槽洗碗机':2400/12,
'嵌入式洗碗机':2400/12,
}
productgroup_map={
    '吸油烟机': ['吸油烟机'],
    '灶具': ['灶具'],
    '蒸烤微合计': ['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶集成': ['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜': ['消毒柜'],
    '热水器': ['热水器','两用炉'],
    '净水机': ['家用净水机','商用净水机'],
    '洗碗机': ['水槽洗碗机','嵌入式洗碗机']
}
productgroup_sort_map = {
    '吸油烟机': 0,
    '灶具': 1,
    '蒸烤微合计': 2,
    '灶集成': 3,
    '消毒柜': 4,
    '热水器': 5,
    '净水机': 6,
    '洗碗机': 7
}
pro_group_type_map={
    '吸油烟机': '吸油烟机',
    '灶具': '灶具',
    '烤箱': '蒸烤微合计',
    '蒸箱': '蒸烤微合计',
    '微波炉': '蒸烤微合计',
    '蒸烤烹饪机': '蒸烤微合计',
    '蒸烤微烹饪机': '蒸烤微合计',
    '蒸微': '蒸烤微合计',
    '灶消烹饪机': '灶集成',
    '灶蒸烹饪机': '灶集成',
    '灶蒸烤烹饪机': '灶集成',
    '灶烤烹饪机': '灶集成',
    '消毒柜': '消毒柜',
    '热水器': '热水器',
    '两用炉': '热水器',
    '家用净水机': '家用净水机',
    '商用净水机': '商用净水机',
    '水槽洗碗机': '水槽/嵌入洗碗机合计',
    '嵌入式洗碗机': '水槽/嵌入洗碗机合计'
}

### 读取单型号贡献报告中整理出的中间数据（处理了物流、财务、渠道、产品组、国内、核算价）

In [45]:
df = pd.read_excel(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\合并物流-财务-产品组-核算价-国内-用于低效-长尾.xlsx')
df['商品编码'] = df['商品编码'].astype(str).map(lambda x: x[:13])
print(df['渠道'].value_counts())
df
# print(len(df))

渠道
零售    461996
工程     19552
电商     16071
Name: count, dtype: int64


,商品编码,渠道,实际出库数量,物料编码,产品型号,产品组,标准型号,国内/海外,产品状态,对应渠道状态,开始销售时间,停止销售时间,系统核算价,核算价
0,1001001500116,零售,12,1001001500116,CXW-358-Z8T(不带罩),吸油烟机,Z8T,国内,停止销售,停止销售,2022-12-10 12:00:00,2025-11-24 16:51:46,3358,40296
1,1009000600033,零售,1,1009000600033,ZK50-02-F1,蒸烤烹饪机,ZK50-02-F1,国内,量产,在售,2025-06-09 12:00:00,NaT,3450,3450
2,1009000500035,零售,3,1009000500035,JZT-ZK46-X2,灶蒸烤烹饪机,JZT-ZK46-X2,国内,量产,在售,2025-04-29 12:00:00,NaT,5280,15840
3,1001001500131,零售,6,1001001500131,CXW-358-02-Z6TA(不带罩),吸油烟机,02-Z6TA,国内,停止销售,停止销售,2024-04-02 12:00:00,2025-11-24 16:51:46,2988,17928
4,1002003700049,零售,5,1002003700049,JZT-01-H8B-12T,灶具,H8B,国内,量产,在售,2024-01-29 12:00:00,NaT,2550,12750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497614,1018001100016,零售,-12,1018001100016,JBCD7E-04-Z5,嵌入式洗碗机,JBCD7E-04-Z5,国内,量产,在售,2025-06-19 12:00:00,NaT,3750,-45000
497615,1018001100017,电商,6,1018001100017,JBCD7E-04-Y1,嵌入式洗碗机,JBCD7E-04-Y1,国内,量产,NaN,NaT,NaT,4500,27000
497616,1018001100017,零售,-6,1018001100017,JBCD7E-04-Y1,嵌入式洗碗机,JBCD7E-04-Y1,国内,量产,在售,2025-08-22 12:00:00,NaT,4500,-27000
497617,1018001100018,电商,6,1018001100018,JBCD7E-04-M-Y1,嵌入式洗碗机,JBCD7E-04-M-Y1,国内,量产,NaN,NaT,NaT,4500,27000


### 将产品的生命周期相关信息匹配进来

In [46]:
df['产品最早开始销售时间'] = df.groupby('物料编码')['开始销售时间'].transform('min')
df

,商品编码,渠道,实际出库数量,物料编码,产品型号,产品组,标准型号,国内/海外,产品状态,对应渠道状态,开始销售时间,停止销售时间,系统核算价,核算价,产品最早开始销售时间
0,1001001500116,零售,12,1001001500116,CXW-358-Z8T(不带罩),吸油烟机,Z8T,国内,停止销售,停止销售,2022-12-10 12:00:00,2025-11-24 16:51:46,3358,40296,2022-12-10 12:00:00
1,1009000600033,零售,1,1009000600033,ZK50-02-F1,蒸烤烹饪机,ZK50-02-F1,国内,量产,在售,2025-06-09 12:00:00,NaT,3450,3450,2025-06-09 12:00:00
2,1009000500035,零售,3,1009000500035,JZT-ZK46-X2,灶蒸烤烹饪机,JZT-ZK46-X2,国内,量产,在售,2025-04-29 12:00:00,NaT,5280,15840,2025-04-29 12:00:00
3,1001001500131,零售,6,1001001500131,CXW-358-02-Z6TA(不带罩),吸油烟机,02-Z6TA,国内,停止销售,停止销售,2024-04-02 12:00:00,2025-11-24 16:51:46,2988,17928,2024-04-02 12:00:00
4,1002003700049,零售,5,1002003700049,JZT-01-H8B-12T,灶具,H8B,国内,量产,在售,2024-01-29 12:00:00,NaT,2550,12750,2024-01-29 12:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497614,1018001100016,零售,-12,1018001100016,JBCD7E-04-Z5,嵌入式洗碗机,JBCD7E-04-Z5,国内,量产,在售,2025-06-19 12:00:00,NaT,3750,-45000,2025-06-19 12:00:00
497615,1018001100017,电商,6,1018001100017,JBCD7E-04-Y1,嵌入式洗碗机,JBCD7E-04-Y1,国内,量产,NaN,NaT,NaT,4500,27000,2025-08-22 12:00:00
497616,1018001100017,零售,-6,1018001100017,JBCD7E-04-Y1,嵌入式洗碗机,JBCD7E-04-Y1,国内,量产,在售,2025-08-22 12:00:00,NaT,4500,-27000,2025-08-22 12:00:00
497617,1018001100018,电商,6,1018001100018,JBCD7E-04-M-Y1,嵌入式洗碗机,JBCD7E-04-M-Y1,国内,量产,NaN,NaT,NaT,4500,27000,2025-08-22 12:00:00


### 选出低效产品

In [47]:
df1 = df.copy()
# 筛选掉新品（新品：最早开始销售时间到现在未满6个月的）
df1 = df1[df1['产品最早开始销售时间']<=pd.Timestamp(cur_date)-pd.DateOffset(months=6)]
# 再在剩下的范围内选出有量产的标准型号
mass_standard = list(df1[df1['产品状态']=='量产']['标准型号'].drop_duplicates())
df1 = df1[df1['标准型号'].isin(mass_standard)].reset_index(drop=True)
df1

,商品编码,渠道,实际出库数量,物料编码,产品型号,产品组,标准型号,国内/海外,产品状态,对应渠道状态,开始销售时间,停止销售时间,系统核算价,核算价,产品最早开始销售时间
0,1009000600033,零售,1,1009000600033,ZK50-02-F1,蒸烤烹饪机,ZK50-02-F1,国内,量产,在售,2025-06-09 12:00:00,NaT,3450,3450,2025-06-09 12:00:00
1,1009000500035,零售,3,1009000500035,JZT-ZK46-X2,灶蒸烤烹饪机,JZT-ZK46-X2,国内,量产,在售,2025-04-29 12:00:00,NaT,5280,15840,2025-04-29 12:00:00
2,1002003700049,零售,5,1002003700049,JZT-01-H8B-12T,灶具,H8B,国内,量产,在售,2024-01-29 12:00:00,NaT,2550,12750,2024-01-29 12:00:00
3,1005000700021,零售,6,1005000700021,KQD62F-02-M1A,烤箱,KQD62F-02-M1A,国内,量产,在售,2024-04-28 12:00:00,NaT,3400,20400,2024-04-28 12:00:00
4,1004001900003,零售,2,1004001900003,JSQ31-X16G2.i-FR-12T,热水器,JSQ31-X16G2.i,国内,量产,在售,2023-02-20 12:00:00,NaT,3050,6100,2023-02-20 12:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
364859,1018001100011,零售,-153,1018001100011,JBCD7E-04-H2,嵌入式洗碗机,JBCD7E-04-H2,国内,量产,在售,2025-06-06 12:00:00,NaT,3150,-481950,2025-06-06 12:00:00
364860,1018001100013,电商,39,1018001100013,JBCD7E-04-H1,嵌入式洗碗机,JBCD7E-04-H1,国内,量产,NaN,NaT,NaT,3750,146250,2025-06-06 12:00:00
364861,1018001100013,零售,-39,1018001100013,JBCD7E-04-H1,嵌入式洗碗机,JBCD7E-04-H1,国内,量产,在售,2025-06-06 12:00:00,NaT,3750,-146250,2025-06-06 12:00:00
364862,1018001100016,电商,12,1018001100016,JBCD7E-04-Z5,嵌入式洗碗机,JBCD7E-04-Z5,国内,量产,NaN,NaT,NaT,3750,45000,2025-06-19 12:00:00


### 对于物料编码进行分组数量汇总聚合

In [48]:
df_calu = df1.groupby('商品编码',as_index=False).agg(
                                                    产品型号=('产品型号','max'),
                                                    产品组=('产品组','max'),
                                                    产品最早开始销售时间=('产品最早开始销售时间','min'),
                                                    产品状态 = ('产品状态','max'),
                                                    标准型号=('标准型号','max'),
                                                    出库数量 = ('实际出库数量','sum'),
                                                    核算价=('核算价','sum'),                                                          
)
print(df_calu.info())
df_calu



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 584 entries, 0 to 583
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   商品编码        584 non-null    object        
 1   产品型号        584 non-null    object        
 2   产品组         584 non-null    object        
 3   产品最早开始销售时间  584 non-null    datetime64[ns]
 4   产品状态        584 non-null    object        
 5   标准型号        584 non-null    object        
 6   出库数量        584 non-null    int64         
 7   核算价         584 non-null    int64         
dtypes: datetime64[ns](1), int64(2), object(5)
memory usage: 36.6+ KB
None


,商品编码,产品型号,产品组,产品最早开始销售时间,产品状态,标准型号,出库数量,核算价
0,1001000200087,CXW-258-EA06,吸油烟机,2018-05-18 12:00:00,量产,EA06,16,310400
1,1001000500324,CXW-258-JQ36(不带罩),吸油烟机,2019-10-15 12:00:00,量产,JQ36,11645,21170610
2,1001000500372,CXW-358-JQ31A(不带罩）,吸油烟机,2021-10-29 12:00:00,量产,JQ31A,10258,19797940
3,1001000500375,CXW-358-JQ33A(不带罩）,吸油烟机,2021-10-29 12:00:00,停止销售,JQ31A,1593,3074490
4,1001000500376,CXW-358-JQ32A(不带罩）,吸油烟机,2021-10-29 12:00:00,停止销售,JQ31A,5050,9746500
...,...,...,...,...,...,...,...,...
579,1018001100004,JBCD7E-03-Y1Pro,嵌入式洗碗机,2024-09-13 12:00:00,量产,JBCD7E-03-Y1.i,22,176000
580,1018001100009,JBCD7E-04-X20,嵌入式洗碗机,2025-06-19 12:00:00,量产,JBCD7E-04-X20,3239,12956000
581,1018001100011,JBCD7E-04-H2,嵌入式洗碗机,2025-06-06 12:00:00,量产,JBCD7E-04-H2,10782,33963300
582,1018001100013,JBCD7E-04-H1,嵌入式洗碗机,2025-06-06 12:00:00,量产,JBCD7E-04-H1,5819,21821250


### 计算出每个产品型号的月均发货数

In [49]:
df_calu['各型号已售卖月份'] = ((pd.Timestamp(cur_date)-df_calu['产品最早开始销售时间']).dt.days/30).round()
df_calu['统计周期内已售卖月份'] = df_calu['各型号已售卖月份'].apply(lambda x: min(x, 12))
# 这里是新的计算方式，老的话，直接用出库数量/12月
df_calu['各型号统计周期内月平均发货量'] = df_calu['出库数量']/df_calu['统计周期内已售卖月份']
df_calu

,商品编码,产品型号,产品组,产品最早开始销售时间,产品状态,标准型号,出库数量,核算价,各型号已售卖月份,统计周期内已售卖月份,各型号统计周期内月平均发货量
0,1001000200087,CXW-258-EA06,吸油烟机,2018-05-18 12:00:00,量产,EA06,16,310400,93.0,12.0,1.333333
1,1001000500324,CXW-258-JQ36(不带罩),吸油烟机,2019-10-15 12:00:00,量产,JQ36,11645,21170610,76.0,12.0,970.416667
2,1001000500372,CXW-358-JQ31A(不带罩）,吸油烟机,2021-10-29 12:00:00,量产,JQ31A,10258,19797940,51.0,12.0,854.833333
3,1001000500375,CXW-358-JQ33A(不带罩）,吸油烟机,2021-10-29 12:00:00,停止销售,JQ31A,1593,3074490,51.0,12.0,132.750000
4,1001000500376,CXW-358-JQ32A(不带罩）,吸油烟机,2021-10-29 12:00:00,停止销售,JQ31A,5050,9746500,51.0,12.0,420.833333
...,...,...,...,...,...,...,...,...,...,...,...
579,1018001100004,JBCD7E-03-Y1Pro,嵌入式洗碗机,2024-09-13 12:00:00,量产,JBCD7E-03-Y1.i,22,176000,16.0,12.0,1.833333
580,1018001100009,JBCD7E-04-X20,嵌入式洗碗机,2025-06-19 12:00:00,量产,JBCD7E-04-X20,3239,12956000,6.0,6.0,539.833333
581,1018001100011,JBCD7E-04-H2,嵌入式洗碗机,2025-06-06 12:00:00,量产,JBCD7E-04-H2,10782,33963300,7.0,7.0,1540.285714
582,1018001100013,JBCD7E-04-H1,嵌入式洗碗机,2025-06-06 12:00:00,量产,JBCD7E-04-H1,5819,21821250,7.0,7.0,831.285714


### 计算出每个标准型号的总发货数，在依据其的产品组标准判断，各个各个标准型号是否是低效整机

In [50]:
df_progroup_stand_grouped = df_calu.groupby(['产品组','标准型号'],as_index=False).agg(
                                                                    标准型号月均发货数 = ('各型号统计周期内月平均发货量','sum')
)
df_progroup_stand_grouped
for index, row in df_progroup_stand_grouped.iterrows():
    if row['标准型号月均发货数'] < Inefficient_standard_map[row['产品组']]:
        df_progroup_stand_grouped.loc[index,'标准型号是否低效'] = '是'
    else:
        df_progroup_stand_grouped.loc[index,'标准型号是否低效'] = '否'
df_progroup_stand_grouped

,产品组,标准型号,标准型号月均发货数,标准型号是否低效
0,两用炉,L1GB32-X06,7.666667,是
1,两用炉,L1PB20-P01,15.250000,是
2,两用炉,L1PB26-P03T,47.666667,是
3,两用炉,L1PB26-P05.i,32.750000,是
4,吸油烟机,01-EMQ5T,76.666667,是
...,...,...,...,...
270,蒸烤烹饪机,ZK72-X20,200.750000,否
271,蒸烤烹饪机,ZK72-Y1.i,68.083333,是
272,蒸箱,SCD45-EX1.i,282.916667,否
273,蒸箱,SCD48-02-M1A,813.750000,否


In [51]:
df_progroup_stand_grouped.to_excel(fr"C:\Users\zhangbon\Desktop\低效明细.xlsx", index=False)

### 产品类别的低效标准型号、总标准型号数

In [52]:
# 找出所有的低效标准型号
list_low = list(df_progroup_stand_grouped[df_progroup_stand_grouped['标准型号是否低效']=='是']['标准型号'])
# list_low

In [53]:
df['产品类别']= df['产品组'].apply(lambda x: pro_group_type_map[x])
df['标准型号是否低效'] = df['标准型号'].apply(lambda x : '是' if x in list_low else '否')
df

,商品编码,渠道,实际出库数量,物料编码,产品型号,产品组,标准型号,国内/海外,产品状态,对应渠道状态,开始销售时间,停止销售时间,系统核算价,核算价,产品最早开始销售时间,产品类别,标准型号是否低效
0,1001001500116,零售,12,1001001500116,CXW-358-Z8T(不带罩),吸油烟机,Z8T,国内,停止销售,停止销售,2022-12-10 12:00:00,2025-11-24 16:51:46,3358,40296,2022-12-10 12:00:00,吸油烟机,否
1,1009000600033,零售,1,1009000600033,ZK50-02-F1,蒸烤烹饪机,ZK50-02-F1,国内,量产,在售,2025-06-09 12:00:00,NaT,3450,3450,2025-06-09 12:00:00,蒸烤微合计,否
2,1009000500035,零售,3,1009000500035,JZT-ZK46-X2,灶蒸烤烹饪机,JZT-ZK46-X2,国内,量产,在售,2025-04-29 12:00:00,NaT,5280,15840,2025-04-29 12:00:00,灶集成,否
3,1001001500131,零售,6,1001001500131,CXW-358-02-Z6TA(不带罩),吸油烟机,02-Z6TA,国内,停止销售,停止销售,2024-04-02 12:00:00,2025-11-24 16:51:46,2988,17928,2024-04-02 12:00:00,吸油烟机,否
4,1002003700049,零售,5,1002003700049,JZT-01-H8B-12T,灶具,H8B,国内,量产,在售,2024-01-29 12:00:00,NaT,2550,12750,2024-01-29 12:00:00,灶具,否
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497614,1018001100016,零售,-12,1018001100016,JBCD7E-04-Z5,嵌入式洗碗机,JBCD7E-04-Z5,国内,量产,在售,2025-06-19 12:00:00,NaT,3750,-45000,2025-06-19 12:00:00,水槽/嵌入洗碗机合计,否
497615,1018001100017,电商,6,1018001100017,JBCD7E-04-Y1,嵌入式洗碗机,JBCD7E-04-Y1,国内,量产,NaN,NaT,NaT,4500,27000,2025-08-22 12:00:00,水槽/嵌入洗碗机合计,否
497616,1018001100017,零售,-6,1018001100017,JBCD7E-04-Y1,嵌入式洗碗机,JBCD7E-04-Y1,国内,量产,在售,2025-08-22 12:00:00,NaT,4500,-27000,2025-08-22 12:00:00,水槽/嵌入洗碗机合计,否
497617,1018001100018,电商,6,1018001100018,JBCD7E-04-M-Y1,嵌入式洗碗机,JBCD7E-04-M-Y1,国内,量产,NaN,NaT,NaT,4500,27000,2025-08-22 12:00:00,水槽/嵌入洗碗机合计,否


In [54]:
def agg_standard_model(x):
    """
    按产品类别聚合统计标准型号相关指标
    x: 每个产品类别分组对应的DataFrame
    返回: 包含各统计指标的Series
    """
    # 低效标准型号数：筛选低效+标准型号去重计数
    inefficient_count = x.loc[x['标准型号是否低效'] == '是', '标准型号'].nunique()
    # 非低效标准型号数：筛选非低效+标准型号去重计数（字段名保留你原有的“占比”，后续可调整）
    non_inefficient_count = x.loc[x['标准型号是否低效'] == '否', '标准型号'].nunique()
    # 标准型号总数：所有标准型号去重计数
    total_count = x['标准型号'].nunique()
    
    # 返回聚合结果，字段名与你原需求一致
    return pd.Series({
        '低效标准型号数': inefficient_count,
        '非低效标准型号数': non_inefficient_count,
        '标准型号总数': total_count
    })

df_result1 = df.groupby('产品类别',as_index=False).apply(agg_standard_model).sort_values(by='产品类别',key=lambda x: x.map(productgroup_sort_map))
df_result1

C:\Users\zhangbon\AppData\Local\Temp\ipykernel_18756\2371401886.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_result1 = df.groupby('产品类别',as_index=False).apply(agg_standard_model).sort_values(by='产品类别',key=lambda x: x.map(productgroup_sort_map))


,产品类别,低效标准型号数,非低效标准型号数,标准型号总数
0,吸油烟机,21,139,160
4,灶具,13,84,97
7,蒸烤微合计,3,57,60
5,灶集成,3,18,21
3,消毒柜,6,21,27
6,热水器,15,47,62
1,家用净水机,1,24,25
2,水槽/嵌入洗碗机合计,13,110,123


### 统计各个渠道的低效产品型号数和产品型号数

In [55]:
df_result2 = pd.DataFrame()
df_result2['产品类别'] = productgroup_map.keys()
for k,v in productgroup_map.items():
    ### 零售
    count_vals1 = df[(df['产品组'].isin(v))&(df['标准型号'].isin(list_low))&(df['渠道']=='零售')]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'零售低效产品型号数量'] = count_vals1
    count_vals2 = df[(df['产品组'].isin(v))&(df['渠道']=='零售')]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'零售产品型号数量'] = count_vals2
    df_result2.loc[df_result2['产品类别'] == k,'零售产品型号数量占比'] =  count_vals1/count_vals2

    ### 工程
    count_vals1 = df[(df['产品组'].isin(v))&(df['标准型号'].isin(list_low))&(df['渠道']=='工程')]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'工程低效产品型号数量'] = count_vals1
    count_vals2 = df[(df['产品组'].isin(v))&(df['渠道']=='工程')]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'工程产品型号数量'] = count_vals2
    df_result2.loc[df_result2['产品类别'] == k,'工程产品型号数量占比'] =  count_vals1/count_vals2

    ### 电商
    count_vals1 = df[(df['产品组'].isin(v))&(df['标准型号'].isin(list_low))&(df['渠道']=='电商')]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'电商低效产品型号数量'] = count_vals1
    count_vals2 = df[(df['产品组'].isin(v))&(df['渠道']=='电商')]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'电商产品型号数量'] = count_vals2
    df_result2.loc[df_result2['产品类别'] == k,'电商产品型号数量占比'] =  count_vals1/count_vals2

    ### 全渠道
    count_vals1 = df[(df['产品组'].isin(v))&(df['标准型号'].isin(list_low))]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'全渠道低效产品型号数量'] = count_vals1
    count_vals2 = df[(df['产品组'].isin(v))]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'全渠道产品型号数量'] = count_vals2
    df_result2.loc[df_result2['产品类别'] == k,'全渠道产品型号数量占比'] =  count_vals1/count_vals2
df_result2


,产品类别,零售低效产品型号数量,零售产品型号数量,零售产品型号数量占比,工程低效产品型号数量,工程产品型号数量,工程产品型号数量占比,电商低效产品型号数量,电商产品型号数量,电商产品型号数量占比,全渠道低效产品型号数量,全渠道产品型号数量,全渠道产品型号数量占比
0,吸油烟机,6.0,118.0,0.050847,16.0,108.0,0.148148,10.0,146.0,0.068493,24.0,243.0,0.098765
1,灶具,21.0,215.0,0.097674,14.0,103.0,0.135922,17.0,197.0,0.086294,30.0,344.0,0.087209
2,蒸烤微合计,2.0,50.0,0.040000,2.0,38.0,0.052632,4.0,57.0,0.070175,4.0,70.0,0.057143
3,灶集成,4.0,68.0,0.058824,4.0,18.0,0.222222,1.0,30.0,0.033333,6.0,72.0,0.083333
4,消毒柜,6.0,35.0,0.171429,5.0,27.0,0.185185,4.0,34.0,0.117647,7.0,51.0,0.137255
5,热水器,15.0,66.0,0.227273,1.0,30.0,0.033333,15.0,57.0,0.263158,18.0,92.0,0.195652
6,净水机,1.0,33.0,0.030303,1.0,18.0,0.055556,0.0,29.0,0.000000,1.0,37.0,0.027027
7,洗碗机,9.0,81.0,0.111111,4.0,58.0,0.068966,11.0,119.0,0.092437,14.0,150.0,0.093333


### 输出标准型号和渠道型号的统计分析

In [56]:

#输出df_qudao和df_calu2，写到一个excel里面
with pd.ExcelWriter(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\低效统计分析结果-26新版.xlsx') as writer:
    df_result1.to_excel(writer, sheet_name='标准型号统计',index=False)
    df_result2.to_excel(writer, sheet_name='分渠道产品型号统计',index=False)

